# Training the Transformer (English → Bangla)

This notebook trains the Transformer we built in `model.ipynb` on English → Bangla translation dataset.

We will go step by step:
1. Load the dataset
2. Build tokenizers (turn words into numbers)
3. Prepare the data for the model
4. Build the model
5. Train the model
6. Test the model by translating sentences



In [23]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, random_split
import pandas as pd
from pathlib import Path

from tokenizers import Tokenizer
from tokenizers.models import WordLevel
from tokenizers.trainers import WordLevelTrainer
from tokenizers.pre_tokenizers import Whitespace

## Load the model we built in `model.ipynb`

`model.ipynb` already contains our Transformer architecture (`BuildTransformer`, etc).
Instead of copy-pasting all that code here, we simply "run" that notebook so all of its
classes become available in this notebook too.


In [28]:
 %run "model.ipynb"

## Step 1: Configuration

All the settings for our training run live in one place: the `config` dictionary.

Our dataset (`docs/en_bn.csv`) 

In [29]:
config = {
    # data
    "dataset_path": "docs/en_bn.csv",
    "src_lang": "en_text",
    "tgt_lang": "bn_text",

    # tokenizer
    "tokenizer_dir": "tokenizers",

    # model size
    "seq_len": 64,
    "d_model": 512,

    "num_encoder_layers": 6,
    "num_decoder_layers": 6,

    "num_heads": 8,

    "d_ff": 2048,

    "dropout": 0.1,

    # training

    "batch_size": 16,
    "num_epochs": 20,

    "lr": 1e-4,

    # where to save trained model checkpoints

    "model_dir": "weights",
    "model_basename": "transformer_model_",

}

## Step 2: Load the dataset and split it into train / validation

We read the CSV file with pandas, drop any empty rows, shuffle it, and split it into a
training set (90%) and a validation set (10%). The validation sentences are never used
for training — we only use them at the end to check how well the model translates
sentences it has not memorized.

In [30]:
def get_dataset(config):

    df = pd.read_csv(config["dataset_path"])

    # Dropping rows with missing values
    df.dropna(inplace=True)

    # Shuffling the dataset
    df = df.sample(frac=1).reset_index(drop=True)

    # Convert DataFrame to a list of plain dictionaries, e.g. {"en_text": ..., "bn_text": ...}
    dataset = list(df.to_dict("records"))

    # 90% train, 10% validation
    train_size = int(0.9 * len(dataset))
    val_size = len(dataset) - train_size

    train_dataset, val_dataset = random_split(
        dataset,
        [train_size, val_size]
    )

    return train_dataset, val_dataset

## Step 3: Build the tokenizers

Neural networks only understand numbers, not words. A **tokenizer** turns text into a
list of numbers (one number per word here, since we use a simple word-level tokenizer).

We build one tokenizer for English and one for Bangla, and save them to disk so we don't
need to rebuild them every time we open this notebook.

In [31]:
def get_or_build_tokenizer(config, ds, lang):

    tokenizer_path = Path(config["tokenizer_dir"]) / f"{lang}_tokenizer.json"

    tokenizer_path.parent.mkdir(
        parents=True,
        exist_ok=True
    )

    if tokenizer_path.exists():

        tokenizer = Tokenizer.from_file(str(tokenizer_path))

        print(f"Loaded {lang} tokenizer from {tokenizer_path}")

    else:

        print(f"Building {lang} tokenizer...")

        tokenizer = Tokenizer(
            WordLevel(unk_token="[UNK]")
        )

        trainer = WordLevelTrainer(
            special_tokens=[
                "[UNK]",  # unknown word (a word the tokenizer has never seen)
                "[PAD]",  # padding, used to make every sentence the same length
                "[SOS]",  # start-of-sentence marker
                "[EOS]"   # end-of-sentence marker
            ]
        )

        tokenizer.pre_tokenizer = Whitespace()

        tokenizer.train_from_iterator(
            ds[lang],
            trainer
        )

        tokenizer.save(str(tokenizer_path))

        print(f"Saved {lang} tokenizer to {tokenizer_path}")

    return tokenizer


def build_tokenizers(config):
    # We build the tokenizers using the FULL dataset (before the train/val split)
    # so that both tokenizers know about every word that appears in the CSV.
    df = pd.read_csv(config["dataset_path"])
    df.dropna(inplace=True)

    tokenizer_src = get_or_build_tokenizer(config, df, config["src_lang"])
    tokenizer_tgt = get_or_build_tokenizer(config, df, config["tgt_lang"])

    return tokenizer_src, tokenizer_tgt

## Step 4: Turn text pairs into tensors the model can train on

For every sentence pair, we need to build three sequences of numbers, all padded to the
same length (`seq_len`):

- **encoder_input**: `[SOS] word word word ... [EOS] [PAD] [PAD] ...` (the English sentence)
- **decoder_input**: `[SOS] word word word ... [PAD] [PAD] ...` (the Bangla sentence, shifted right)
- **label**: `word word word ... [EOS] [PAD] [PAD] ...` (the Bangla sentence the model should predict)

We also build two **masks**:
- `encoder_mask` hides the `[PAD]` tokens so the model ignores them.
- `decoder_mask` hides `[PAD]` tokens **and** hides future words, so that while predicting
  word number 3, the model can only look at words 1 and 2 (not "cheat" by looking ahead).

In [32]:
def causal_mask(size):
    # Returns a (1, size, size) mask that is 1 on and below the diagonal, 0 above it.
    # "1" means "allowed to look at this position", "0" means "not allowed" (future word).
    mask = torch.triu(torch.ones((1, size, size)), diagonal=1).type(torch.int)
    return (mask == 0).int()

In [33]:
class BilingualDataset(Dataset):
    def __init__(self, dataset, tokenizer_src, tokenizer_tgt, src_lang, tgt_lang, seq_len):
        super().__init__()
        self.dataset = dataset
        self.tokenizer_src = tokenizer_src
        self.tokenizer_tgt = tokenizer_tgt
        self.src_lang = src_lang
        self.tgt_lang = tgt_lang
        self.seq_len = seq_len

        self.sos_token = torch.tensor([tokenizer_tgt.token_to_id("[SOS]")], dtype=torch.int64)
        self.eos_token = torch.tensor([tokenizer_tgt.token_to_id("[EOS]")], dtype=torch.int64)
        self.pad_token = torch.tensor([tokenizer_tgt.token_to_id("[PAD]")], dtype=torch.int64)

    def __len__(self):
        return len(self.dataset)

    def __getitem__(self, idx):
        pair = self.dataset[idx]
        src_text = pair[self.src_lang]
        tgt_text = pair[self.tgt_lang]

        # turn the sentences into lists of token ids
        enc_input_tokens = self.tokenizer_src.encode(src_text).ids
        dec_input_tokens = self.tokenizer_tgt.encode(tgt_text).ids

        # how many [PAD] tokens we need to reach seq_len
        enc_num_padding = self.seq_len - len(enc_input_tokens) - 2  # -2 for [SOS] and [EOS]
        dec_num_padding = self.seq_len - len(dec_input_tokens) - 1  # -1 for [SOS] only

        if enc_num_padding < 0 or dec_num_padding < 0:
            raise ValueError(f"Sentence is too long for seq_len={self.seq_len}, increase seq_len in config")

        # encoder input: [SOS] ... english sentence ... [EOS] [PAD]...
        encoder_input = torch.cat([
            self.sos_token,
            torch.tensor(enc_input_tokens, dtype=torch.int64),
            self.eos_token,
            torch.tensor([self.pad_token.item()] * enc_num_padding, dtype=torch.int64)
        ])

        # decoder input: [SOS] ... bangla sentence ... [PAD]...  (no [EOS] here)
        decoder_input = torch.cat([
            self.sos_token,
            torch.tensor(dec_input_tokens, dtype=torch.int64),
            torch.tensor([self.pad_token.item()] * dec_num_padding, dtype=torch.int64)
        ])

        # label: ... bangla sentence ... [EOS] [PAD]...  (what the decoder should predict)
        label = torch.cat([
            torch.tensor(dec_input_tokens, dtype=torch.int64),
            self.eos_token,
            torch.tensor([self.pad_token.item()] * dec_num_padding, dtype=torch.int64)
        ])

        assert encoder_input.size(0) == self.seq_len
        assert decoder_input.size(0) == self.seq_len
        assert label.size(0) == self.seq_len

        # mask out [PAD] tokens for the encoder
        encoder_mask = (encoder_input != self.pad_token.item()).unsqueeze(0).unsqueeze(0).int()  # (1, 1, seq_len)

        # mask out [PAD] tokens AND future tokens for the decoder
        decoder_padding_mask = (decoder_input != self.pad_token.item()).unsqueeze(0).unsqueeze(0).int()  # (1, 1, seq_len)
        decoder_mask = decoder_padding_mask & causal_mask(decoder_input.size(0))  # (1, seq_len, seq_len)

        return {
            "encoder_input": encoder_input,   # (seq_len)
            "decoder_input": decoder_input,   # (seq_len)
            "encoder_mask": encoder_mask,     # (1, 1, seq_len)
            "decoder_mask": decoder_mask,     # (1, seq_len, seq_len)
            "label": label,                   # (seq_len)
            "src_text": src_text,
            "tgt_text": tgt_text
        }

## Step 5: Create the DataLoaders

A `DataLoader` groups our data into small batches and shuffles it for us, so we don't
have to do that by hand in the training loop.

In [34]:
def get_dataloaders(config):
    tokenizer_src, tokenizer_tgt = build_tokenizers(config)
    train_dataset, val_dataset = get_dataset(config)

    train_ds = BilingualDataset(train_dataset, tokenizer_src, tokenizer_tgt, config["src_lang"], config["tgt_lang"], config["seq_len"])
    val_ds = BilingualDataset(val_dataset, tokenizer_src, tokenizer_tgt, config["src_lang"], config["tgt_lang"], config["seq_len"])

    train_dataloader = DataLoader(train_ds, batch_size=config["batch_size"], shuffle=True)
    # batch_size=1 for validation, since we translate and print one sentence at a time
    val_dataloader = DataLoader(val_ds, batch_size=1, shuffle=False)

    return train_dataloader, val_dataloader, tokenizer_src, tokenizer_tgt

## Step 6: Build the model

`BuildTransformer` comes from `model.ipynb`. We just need to tell it how big our
vocabularies are (how many different words each tokenizer knows).

In [35]:
def build_model(config, src_vocab_size, tgt_vocab_size):
    builder = BuildTransformer(
        num_encoder_layers=config["num_encoder_layers"],
        num_decoder_layers=config["num_decoder_layers"],
        d_model=config["d_model"],
        num_heads=config["num_heads"],
        d_ff=config["d_ff"],
        input_vocab_size=src_vocab_size,
        output_vocab_size=tgt_vocab_size,
        max_seq_length=config["seq_len"],
        dropout=config["dropout"]
    )
    return builder.get_model()

## Step 7: Train the model

For every batch we:
1. Run the sentences through the model to get predictions
2. Compare the predictions to the real translation (the `label`) using a loss function
3. Adjust the model's weights a tiny bit to reduce that loss (`backward()` + `step()`)

We repeat this for every batch, for `num_epochs` full passes over the training data,
and print the average loss after each epoch so we can see the model improving.

In [36]:
def train_model(config):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Using device: {device}")

    Path(config["model_dir"]).mkdir(parents=True, exist_ok=True)

    train_dataloader, val_dataloader, tokenizer_src, tokenizer_tgt = get_dataloaders(config)

    model = build_model(config, tokenizer_src.get_vocab_size(), tokenizer_tgt.get_vocab_size()).to(device)

    optimizer = torch.optim.Adam(model.parameters(), lr=config["lr"], eps=1e-9)

    pad_token_id = tokenizer_tgt.token_to_id("[PAD]")
    # ignore_index tells the loss to skip [PAD] tokens (they don't count as real words)
    loss_fn = nn.CrossEntropyLoss(ignore_index=pad_token_id, label_smoothing=0.1).to(device)

    for epoch in range(config["num_epochs"]):
        model.train()
        total_loss = 0.0

        for batch in train_dataloader:
            encoder_input = batch["encoder_input"].to(device)   # (batch, seq_len)
            decoder_input = batch["decoder_input"].to(device)   # (batch, seq_len)
            encoder_mask = batch["encoder_mask"].to(device)     # (batch, 1, 1, seq_len)
            decoder_mask = batch["decoder_mask"].to(device)     # (batch, 1, seq_len, seq_len)
            label = batch["label"].to(device)                   # (batch, seq_len)

            # forward pass: predict the Bangla translation
            output = model(encoder_input, decoder_input, encoder_mask, decoder_mask)
            # output: (batch, seq_len, vocab_size)

            # compare predictions to the real translation
            loss = loss_fn(output.view(-1, output.size(-1)), label.view(-1))

            # update the model's weights
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            total_loss += loss.item()

        avg_loss = total_loss / len(train_dataloader)
        if (epoch + 1) % 5 == 0 or epoch == 0:
            print(f"Epoch {epoch + 1}/{config["num_epochs"]} - loss: {avg_loss:.4f}")

    # save the trained model so we don't have to retrain it every time
    final_path = Path(config["model_dir"]) / f"{config["model_basename"]}final.pt"
    torch.save({"model_state_dict": model.state_dict()}, final_path)
    print(f"Saved trained model to {final_path}")

    return model, tokenizer_src, tokenizer_tgt, val_dataloader, device

## Step 8: Translate sentences (test the model)

To translate a sentence, we can't just ask the model for the whole output at once,
because the decoder needs to see its *own* previous words to predict the next one.
So we generate the translation **one word at a time**:

1. Start with just `[SOS]`
2. Ask the model to predict the next word
3. Add that word to the sentence and repeat
4. Stop when the model predicts `[EOS]`, or we hit the max length

This is called **greedy decoding**, because at each step we simply pick the single
most likely next word.

In [37]:
def greedy_decode(model, source, source_mask, tokenizer_src, tokenizer_tgt, max_len, device):
    sos_idx = tokenizer_tgt.token_to_id("[SOS]")
    eos_idx = tokenizer_tgt.token_to_id("[EOS]")

    # encode the source sentence just once
    encoder_output = model.encode(source, source_mask)

    # start the translation with only the [SOS] token
    decoder_input = torch.empty(1, 1).fill_(sos_idx).type_as(source).to(device)

    while True:
        if decoder_input.size(1) == max_len:
            break

        # build a fresh causal mask for the current length of the decoder input
        decoder_mask = causal_mask(decoder_input.size(1)).type_as(source_mask).to(device)

        out = model.decode(decoder_input, encoder_output, source_mask, decoder_mask)

        # look only at the prediction for the last position, and pick the most likely word
        prob = model.projection_to_vocab(out[:, -1])
        _, next_word = torch.max(prob, dim=1)

        decoder_input = torch.cat(
            [decoder_input, torch.empty(1, 1).type_as(source).fill_(next_word.item()).to(device)], dim=1
        )

        if next_word.item() == eos_idx:
            break

    return decoder_input.squeeze(0)


def run_validation(model, val_dataloader, tokenizer_src, tokenizer_tgt, seq_len, device, num_examples=3):
    model.eval()
    count = 0

    with torch.no_grad():
        for batch in val_dataloader:
            count += 1

            encoder_input = batch["encoder_input"].to(device)
            encoder_mask = batch["encoder_mask"].to(device)

            assert encoder_input.size(0) == 1, "Validation dataloader must use batch_size=1"

            model_out = greedy_decode(model, encoder_input, encoder_mask, tokenizer_src, tokenizer_tgt, seq_len, device)

            source_text = batch["src_text"][0]
            target_text = batch["tgt_text"][0]
            predicted_text = tokenizer_tgt.decode(model_out.detach().cpu().numpy())

            print("-" * 40)
            print(f"SOURCE:    {source_text}")
            print(f"TARGET:    {target_text}")
            print(f"PREDICTED: {predicted_text}")

            if count == num_examples:
                break


def translate(model, sentence, tokenizer_src, tokenizer_tgt, seq_len, device):
    """Translate any English sentence you type in, not just ones from the dataset."""
    model.eval()

    sos_id = tokenizer_src.token_to_id("[SOS]")
    eos_id = tokenizer_src.token_to_id("[EOS]")
    pad_id = tokenizer_src.token_to_id("[PAD]")

    src_tokens = tokenizer_src.encode(sentence).ids
    num_padding = seq_len - len(src_tokens) - 2
    if num_padding < 0:
        raise ValueError(f"Sentence is too long for seq_len={seq_len}")

    source = torch.cat([
        torch.tensor([sos_id]),
        torch.tensor(src_tokens, dtype=torch.int64),
        torch.tensor([eos_id]),
        torch.tensor([pad_id] * num_padding, dtype=torch.int64)
    ]).unsqueeze(0).to(device)

    source_mask = (source != pad_id).unsqueeze(0).unsqueeze(0).int().to(device)

    with torch.no_grad():
        model_out = greedy_decode(model, source, source_mask, tokenizer_src, tokenizer_tgt, seq_len, device)

    return tokenizer_tgt.decode(model_out.detach().cpu().numpy())

## Step 9: Run it all

This trains the model on our dataset and then shows a few example translations
from the validation set.


In [38]:
model, tokenizer_src, tokenizer_tgt, val_dataloader, device = train_model(config)

Using device: cuda
Building en_text tokenizer...
Saved en_text tokenizer to tokenizers\en_text_tokenizer.json
Building bn_text tokenizer...
Saved bn_text tokenizer to tokenizers\bn_text_tokenizer.json
Epoch 1/20 - loss: 0.9012
Epoch 5/20 - loss: 0.8143
Epoch 10/20 - loss: 0.8173
Epoch 15/20 - loss: 0.8145
Epoch 20/20 - loss: 0.8159
Saved trained model to weights\transformer_model_final.pt


In [39]:
run_validation(model, val_dataloader, tokenizer_src, tokenizer_tgt, config["seq_len"], device, num_examples=3)

----------------------------------------
SOURCE:    You're welcome!
TARGET:    আপনার স্বাগতম!
PREDICTED: আপনার স্বাগতম !
----------------------------------------
SOURCE:    Greetings, it is an honor to meet you.
TARGET:    শুভেচ্ছা, আপনাকে সম্মান জানাতে পেরে আমি আনন্দিত।
PREDICTED: শুভেচ্ছা , আপনাকে সম্মান জানাতে পেরে আমি আনন্দিত ।
----------------------------------------
SOURCE:    Thank you very much for your help.
TARGET:    আপনার সহায়তার জন্য অনেক ধন্যবাদ।
PREDICTED: আপনার সহায়তার জন্য অনেক ধন্যবাদ ।


## Try your own sentence

In [40]:
translate(model, "Thank you very much for your help.", tokenizer_src, tokenizer_tgt, config["seq_len"], device)

'আপনার সহায়তার জন্য অনেক ধন্যবাদ ।'